# Bronze Data Profiling & Quality Assessment

This notebook profiles the Bronze layer of the Olist E-Commerce Lakehouse.

## Objectives

- Validate Bronze table availability
- Inspect schemas and record counts
- Identify duplicate records
- Measure missing values
- Validate primary/business keys
- Detect invalid numerical values
- Evaluate referential integrity
- Identify timestamp and data quality issues
- Define transformation rules for the Silver layer

In [0]:
from pyspark.sql import functions as F

CATALOG = "ecommerce_lakehouse"
BRONZE_SCHEMA = "bronze"

TABLES = [
    "customers",
    "geolocation",
    "order_items",
    "order_payments",
    "order_reviews",
    "orders",
    "products",
    "sellers",
    "product_category_translation"
]

print(f"Catalog: {CATALOG}")
print(f"Schema : {BRONZE_SCHEMA}")
print(f"Tables : {len(TABLES)}")

Catalog: ecommerce_lakehouse
Schema : bronze
Tables : 9


In [0]:
available_tables = {
    row.tableName
    for row in spark.sql(
        f"SHOW TABLES IN {CATALOG}.{BRONZE_SCHEMA}"
    ).collect()
}

missing_tables = set(TABLES) - available_tables

if missing_tables:
    raise ValueError(
        f"Missing Bronze tables: {sorted(missing_tables)}"
    )

print(
    f"Validation successful. "
    f"All {len(TABLES)} Bronze tables are available."
)

Validation successful. All 9 Bronze tables are available.


In [0]:
row_counts = []

for table_name in TABLES:

    df = spark.table(
        f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"
    )

    row_counts.append({
        "table": table_name,
        "row_count": df.count(),
        "column_count": len(df.columns)
    })

row_counts_df = spark.createDataFrame(row_counts)

display(
    row_counts_df.orderBy(F.desc("row_count"))
)

column_count,row_count,table
7,1000163,geolocation
9,112650,order_items
7,103886,order_payments
7,99441,customers
10,99441,orders
9,99224,order_reviews
11,32951,products
6,3095,sellers
4,71,product_category_translation


In [0]:
for table_name in TABLES:

    print("\n" + "=" * 70)
    print(f"TABLE: {table_name}")
    print("=" * 70)

    spark.table(
        f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"
    ).printSchema()


TABLE: customers
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)


TABLE: geolocation
root
 |-- geolocation_zip_code_prefix: integer (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)


TABLE: order_items
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable

In [0]:
def profile_nulls(table_name):

    df = spark.table(
        f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"
    )

    total_rows = df.count()

    expressions = []

    for column_name in df.columns:

        if column_name.startswith("_"):
            continue

        expressions.append(
            F.sum(
                F.when(
                    F.col(column_name).isNull(),
                    1
                ).otherwise(0)
            ).alias(column_name)
        )

    result = df.agg(*expressions).collect()[0].asDict()

    output = []

    for column_name, null_count in result.items():

        null_percentage = (
            (null_count / total_rows) * 100
            if total_rows > 0
            else 0
        )

        output.append({
            "table": table_name,
            "column": column_name,
            "null_count": null_count,
            "null_percentage": round(null_percentage, 2)
        })

    return output

In [0]:
null_results = []

for table_name in TABLES:
    null_results.extend(
        profile_nulls(table_name)
    )

null_df = spark.createDataFrame(null_results)

display(
    null_df
    .filter(F.col("null_count") > 0)
    .orderBy(
        F.desc("null_percentage")
    )
)

column,null_count,null_percentage,table
review_comment_title,87656,88.34,order_reviews
review_comment_message,58247,58.7,order_reviews
order_delivered_customer_date,2965,2.98,orders
product_category_name,610,1.85,products
product_name_lenght,610,1.85,products
product_description_lenght,610,1.85,products
product_photos_qty,610,1.85,products
order_delivered_carrier_date,1783,1.79,orders
order_approved_at,160,0.16,orders
product_weight_g,2,0.01,products


In [0]:
KEYS = {
    "customers": ["customer_id"],
    "orders": ["order_id"],
    "order_items": ["order_id", "order_item_id"],
    "order_payments": ["order_id", "payment_sequential"],
    "order_reviews": ["review_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "product_category_translation": ["product_category_name"]
}

In [0]:
duplicate_results = []

for table_name, key_columns in KEYS.items():

    df = spark.table(
        f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"
    )

    duplicate_groups = (
        df.groupBy(*key_columns)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    duplicate_results.append({
        "table": table_name,
        "key": ", ".join(key_columns),
        "duplicate_key_groups": duplicate_groups
    })

duplicate_df = spark.createDataFrame(
    duplicate_results
)

display(duplicate_df)

duplicate_key_groups,key,table
0,customer_id,customers
0,order_id,orders
0,"order_id, order_item_id",order_items
0,"order_id, payment_sequential",order_payments
789,review_id,order_reviews
0,product_id,products
0,seller_id,sellers
0,product_category_name,product_category_translation


In [0]:
orders_df = spark.table(
    f"{CATALOG}.{BRONZE_SCHEMA}.orders"
)

customers_df = spark.table(
    f"{CATALOG}.{BRONZE_SCHEMA}.customers"
)

orphan_orders = (
    orders_df
    .join(
        customers_df.select("customer_id"),
        on="customer_id",
        how="left_anti"
    )
)

print(
    "Orders without matching customer:",
    orphan_orders.count()
)

Orders without matching customer: 0


In [0]:
order_items_df = spark.table(
    f"{CATALOG}.{BRONZE_SCHEMA}.order_items"
)

orphan_order_items = (
    order_items_df
    .join(
        orders_df.select("order_id"),
        on="order_id",
        how="left_anti"
    )
)

print(
    "Order items without matching order:",
    orphan_order_items.count()
)

Order items without matching order: 0


In [0]:
products_df = spark.table(
    f"{CATALOG}.{BRONZE_SCHEMA}.products"
)

orphan_products = (
    order_items_df
    .join(
        products_df.select("product_id"),
        on="product_id",
        how="left_anti"
    )
)

print(
    "Order items without matching product:",
    orphan_products.count()
)

Order items without matching product: 0


In [0]:
sellers_df = spark.table(
    f"{CATALOG}.{BRONZE_SCHEMA}.sellers"
)

orphan_sellers = (
    order_items_df
    .join(
        sellers_df.select("seller_id"),
        on="seller_id",
        how="left_anti"
    )
)

print(
    "Order items without matching seller:",
    orphan_sellers.count()
)

Order items without matching seller: 0


In [0]:
invalid_order_items = (
    order_items_df
    .filter(
        (F.col("price") < 0) |
        (F.col("freight_value") < 0)
    )
)

print(
    "Order items with invalid monetary values:",
    invalid_order_items.count()
)

Order items with invalid monetary values: 0


In [0]:
payments_df = spark.table(
    f"{CATALOG}.{BRONZE_SCHEMA}.order_payments"
)

invalid_payments = (
    payments_df
    .filter(
        F.col("payment_value") < 0
    )
)

print(
    "Payments with negative values:",
    invalid_payments.count()
)

Payments with negative values: 0


In [0]:
display(
    orders_df
    .groupBy("order_status")
    .count()
    .orderBy(F.desc("count"))
)

order_status,count
delivered,96478
shipped,1107
canceled,625
unavailable,609
invoiced,314
processing,301
created,5
approved,2
